# Exploratory Data Analysis: Hotel Demand & Occupancy Dynamics

**Project:** Hotel Occupancy & Room-Demand Forecasting  
**Scope:** Multi-Year Booking Ledger Analysis (2023–2026), Booking Pace, Seasonality, and Lead-Time Dynamics.  
**Author:** Candidate (AI & Machine Learning Assessment)

In [1]:

import pandas as pd

# Load cleaned booking ledger
df = pd.read_csv('../data/processed/booking_ledger_clean.csv')
df['booking_date'] = pd.to_datetime(df['booking_date'])
df['arrival_date'] = pd.to_datetime(df['arrival_date'])
df['departure_date'] = pd.to_datetime(df['departure_date'])
df['lead_time'] = (df['arrival_date'] - df['booking_date']).dt.days
print(f'Cleaned ledger loaded: {len(df):,} transactions.')

## 1. Data Quality & Cleaning Audit

Before modeling, data integrity was audited with `validate_ledger`:  
- **Raw Records:** 40,891 | **Cleaned Records:** 40,571  
- **Corrupted Records Purged:** 320 (duplicate IDs, arrival < booking, departure <= arrival).  
- **Imputed:** 123 missing distribution channels imputed with modal class (`OTA`).  
- **Outliers:** 122 ADR values clipped/imputed to category median.  

> **One-line takeaway:** *The cleaning pipeline successfully resolved 100% of detected integrity anomalies, yielding a high-fidelity, leak-free transactional ledger.*

## 2. Occupancy Over Time by Room Category

![Occupancy Over Time](../reports/figures/eda_01_occupancy_over_time.png)

> **One-line takeaway:** *Standard rooms drive high-volume baseline occupancy, while Suites exhibit lower, spikier holiday surges; clear annual peaks occur during winter (Nov–Feb) and troughs during monsoon (Jun–Aug).*

## 3. Weekday and Monthly Seasonality

![Seasonality Dynamics](../reports/figures/eda_02_seasonality.png)

> **One-line takeaway:** *Corporate demand drives midweek occupancy for Standard rooms, while leisure demand generates Friday-Saturday surges for Deluxe and Family suites; annual seasonality shows robust winter peaks.*

## 4. Public Holiday Demand Lift

![Holiday Effect](../reports/figures/eda_03_holiday_effect.png)

> **One-line takeaway:** *Indian gazetted public holidays induce a statistically significant +25% to +35% demand lift across all room categories, with Family and Deluxe categories benefiting most.*

## 5. Lead-Time Distributions by Channel

![Lead-Time Distribution](../reports/figures/eda_04_lead_time_distribution.png)

> **One-line takeaway:** *Lead times follow a heavy right-skewed Gamma distribution; Corporate bookings concentrate tightly around 0–10 days, while OTA leisure bookings exhibit wide windows extending beyond 30 days.*

## 6. Cancellation Rate vs. Lead Time and Channel

![Cancellation Dynamics](../reports/figures/eda_05_cancellation_dynamics.png)

> **One-line takeaway:** *Cancellation risk increases monotonically with lead time and peaks in the OTA channel (>30%), proving that unadjusted bookings severely overestimate final occupancy.*

## 7. Booking Pace Curves (On-The-Books vs. Days Before Arrival)

![Booking Pace Curve](../reports/figures/eda_06_booking_pace_curve.png)

> **One-line takeaway:** *At horizon h=14, approximately 45% of final rooms are already on the books, accelerating to 85% by h=3, providing strong empirical justification for pickup velocity features ($OTB_T - OTB_{T-7}$).*

## 8. Occupancy Autocorrelation (ACF)

![Autocorrelation Plot](../reports/figures/eda_07_autocorrelation.png)

> **One-line takeaway:** *Pronounced autocorrelation spikes at lags 7, 14, 21, and 28 mathematically validate the choice of same-weekday realized occupancy as primary lag features.*

## 9. EDA Conclusions that Changed My Modelling Decisions

1. **Direct Multi-Horizon Topology:** Because booking pace evolves non-linearly across the 14-day window (45% booked at $h=14$ vs. 85% at $h=3$), the forecast horizon $h$ must be an explicit, interactive feature so the tree models learn horizon-dependent pickup decay.
2. **Category-Specific Feature Interactions:** Standard and Family rooms have opposing weekly shapes (midweek vs. weekend). Room category cannot merely be an additive intercept; models must support category-calendar interaction terms.
3. **Horizon-Aware Same-Weekday Lags:** Given the 7-day periodicity shown in the ACF plot, naive lags like $t-1$ are unobservable at $h=14$ and cause leakage. Lags must be indexed as $s - 7 \cdot \lceil h/7 \rceil$ to guarantee they reflect only realized same-weekday history as-of origin $T$.
4. **Pickup Pace Acceleration:** The steepening slope of booking pace inside 14 days proves that static OTB is insufficient; the velocity of bookings ($OTB_T - OTB_{T-7}$) is critical for detecting demand inflection points.
5. **Physical Capacity Clipping:** Realized room nights are strictly bounded by physical capacity ($[0, \text{capacity}_c]$). Model predictions and prediction intervals must be clipped to ensure physically viable operational outputs.